# exp010 NB5: Noisy Student Multi-Round Training (Perch embeddings)

Trains ProtoSSM + MLP head over 5 rounds of pseudo-labeling on the 10,592 unlabeled soundscapes.

- **Round 0**: train on 66 labeled SS files only (= NB4 baseline)
- **Round 1-4**: predict pseudo labels for unlabeled SS using current ensemble → retrain on (real + pseudo)

Output: per-round weight tensors saved to `/kaggle/working/`. Inference handled by NB6.

GPU T4, ~1-2 hours.

In [ ]:
import subprocess, sys, os, time

START = time.time()

# Find perch-onnx dataset
ONNX_DS = None
for _c in [
    "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026",
    "/kaggle/input/perch-onnx-for-birdclef-2026",
    "/kaggle/input/perch-onnx-for-birdclef2026",
]:
    if os.path.isdir(_c):
        ONNX_DS = _c
        break

if ONNX_DS is None:
    print("Available /kaggle/input/:")
    for d in sorted(os.listdir("/kaggle/input/")):
        print(f"  {d}")
        sub = os.path.join("/kaggle/input", d)
        if os.path.isdir(sub):
            for f in sorted(os.listdir(sub))[:5]:
                print(f"    {f}")
    raise FileNotFoundError("perch-onnx dataset not found")

print(f"ONNX dataset: {ONNX_DS}")
print(f"Files: {os.listdir(ONNX_DS)}")

# Install onnxruntime from dataset wheel
whls = [f for f in os.listdir(ONNX_DS) if f.endswith(".whl")]
if whls:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           os.path.join(ONNX_DS, whls[0])])
    print(f"Installed: {whls[0]}")
else:
    print("No wheel found, using pre-installed onnxruntime")

In [ ]:
import gc, re, warnings, glob, random, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(SEED)
print(f"torch {torch.__version__}, device={DEVICE}, seed={SEED}")

In [ ]:
# CONFIG (NB3 v29 + NB2 v10 統合)
SR = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
N_WINDOWS = 12

BASE = Path("/kaggle/input/competitions/birdclef-2026")
if not BASE.exists():
    BASE = Path("/kaggle/input/birdclef-2026")

EMB_DIR = Path("/kaggle/input/notebooks/maekeso/birdclef2026-exp010-nb1-embedding")
ANURA_DIR = Path("/kaggle/input/datasets/maekeso/birdclef2026-perch-embed-anura")
INAT_DIR  = Path("/kaggle/input/datasets/maekeso/birdclef2026-perch-embed-inat-nonbird")
TEST_DIR = BASE / "test_soundscapes"
TRAIN_SC_DIR = BASE / "train_soundscapes"
TAXONOMY_CSV = BASE / "taxonomy.csv"
SC_LABELS_CSV = BASE / "train_soundscapes_labels.csv"

# ONNX_DS is set in install cell
LABELS_CSV = os.path.join(ONNX_DS, "labels.csv")
ONNX_MODEL = os.path.join(ONNX_DS, "perch_v2.onnx")

# ProtoSSM config (NB3 v29)
D_MODEL = 128
D_STATE = 16
N_SSM_LAYERS = 2

# MLP Head config (NB2 v10)
MLP_HIDDEN = 256

# Common training hyperparams
DROPOUT = 0.1
N_EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 15
BATCH_ONNX = 48

# Metadata embedding (site/hour parsed from filename)
N_SITES = 32
META_DIM = 8

# Multi-seed ensemble
SEEDS = [42, 123, 777, 2024, 9999]

# v4 = v2 base (SWA on) + Knowledge Distillation.
# v3 (Focal γ=2.0 単独) LB 0.920 → revert. Focal は pos_weight と過剰圧力で悪化。
# SWA は v2 で ±0.000 だったが v2 を baseline と決めたので維持。
USE_SWA = True
SWA_START_FRAC = 0.65

# Knowledge Distillation (v4): Perch の sigmoid output を soft target として
# loss = BCE(out, hard) + LAMBDA_KD * BCE(out, sigmoid(perch_logit))
# 66 files の partial label noise を Perch baseline で正則化する目的。
LAMBDA_KD = 0.15

# Aggregation across (TTA x seeds)
AGG_MODE = "mean"
BLEND_ALPHA = 0.5

# TTA shifts
TTA_SHIFTS = [-1, 0, 1]

# Prior tables (site/hour co-occurrence)
LAMBDA_PRIOR = 0.3
PRIOR_STRENGTH_SITE = 8.0
PRIOR_STRENGTH_HOUR = 8.0
PRIOR_STRENGTH_SH = 4.0

# Hierarchical Site-Conditioned KNN retrieval (inference-only)
RETRIEVAL_K = 10
RETRIEVAL_TAU = 0.05
RETRIEVAL_ALPHA_SITE = 1.5
RETRIEVAL_ALPHA_HOUR = 1.2
LAMBDA_RETRIEVAL = 0.10
RETRIEVAL_EPS = 1e-4

# BLEND weights (logit space, sum=1.0 not required but kept symmetric for first try)
W_PROTO = 0.5
W_MLP = 0.5

# file_confidence_scale
FCS_TOP_K = 2
FCS_POWER = 0.4

# Train audio retrieval pool (v6): expand soundscape pool (792) → TA pool (265k)
USE_TA_RETRIEVAL = True
RETRIEVAL_TA_K = 20
# v10: class-specific LAMBDA — Aves keeps 0.05, non-Aves boosted 3x to 0.15
# (external non-Aves pool {AnuraSet, iNat} added in v9 had ±0.000 effect with uniform 0.05)
LAMBDA_RETRIEVAL_TA = 0.05         # legacy scalar (kept for fallback / Aves)
LAMBDA_TA_AVES     = 0.05
LAMBDA_TA_NONAVES  = 0.15

META_PAT = re.compile(r"_S(\d{2})_(\d{8})_(\d{2})\d{4}")

print(f"BASE: {BASE}")
print(f"EMB_DIR: {EMB_DIR}")
print(f"Files: {sorted(os.listdir(EMB_DIR))}")

In [ ]:
# TAXONOMY & PERCH LABEL MAPPING
taxonomy = pd.read_csv(TAXONOMY_CSV)
PRIMARY_LABELS = sorted(taxonomy["primary_label"].tolist())
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

bc_labels = (
    pd.read_csv(LABELS_CSV)
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)
NO_LABEL_INDEX = len(bc_labels)

taxonomy_m = taxonomy.copy()
taxonomy_m["scientific_name_lookup"] = taxonomy_m["scientific_name"]
bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})

mapping = taxonomy_m.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup", how="left",
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)
label_to_bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC = BC_INDICES[MAPPED_MASK].astype(np.int32)

proxy_map = {}
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sono = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
]
for _, row in unmapped_non_sono.iterrows():
    genus = str(row["scientific_name"]).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(
            rf"^{re.escape(genus)}\s", na=False
        )
    ]
    if len(hits) > 0:
        proxy_map[label_to_idx[row["primary_label"]]] = (
            hits["bc_index"].astype(int).values
        )

print(f"Species: {N_CLASSES}, Mapped: {MAPPED_MASK.sum()}, Proxies: {len(proxy_map)}")

# v10: class-specific LAMBDA_TA vector — non-Aves species get 3x boost
class_name_arr = np.array([
    taxonomy.set_index("primary_label").loc[lbl, "class_name"]
    for lbl in PRIMARY_LABELS
])
NON_AVES_MASK = (class_name_arr != "Aves")
LAMBDA_TA_VEC = np.full(N_CLASSES, LAMBDA_TA_AVES, dtype=np.float32)
LAMBDA_TA_VEC[NON_AVES_MASK] = LAMBDA_TA_NONAVES
print(f"LAMBDA_TA: Aves={LAMBDA_TA_AVES} ({(~NON_AVES_MASK).sum()} sp), "
      f"non-Aves={LAMBDA_TA_NONAVES} ({NON_AVES_MASK.sum()} sp)")

In [ ]:
# LOAD ALL SOUNDSCAPE EMBEDDINGS (labeled + unlabeled)
sc_data = np.load(EMB_DIR / "soundscape_embeddings.npz")
sc_emb = sc_data["embeddings"].astype(np.float32)
sc_scores = sc_data["scores"].astype(np.float32)
sc_meta = pd.read_parquet(EMB_DIR / "soundscape_meta.parquet")

print(f"All SS: {sc_emb.shape[0]} windows, {sc_meta['filename'].nunique()} files")

sc_labels_df = pd.read_csv(SC_LABELS_CSV)
labeled_files = set(sc_labels_df["filename"].unique())

label_map = {}
for _, r in sc_labels_df.iterrows():
    fn = r["filename"]
    end_sec = int(pd.Timedelta(r["end"]).total_seconds())
    row_id = f"{Path(fn).stem}_{end_sec}"
    labels_str = str(r["primary_label"]).split(";")
    y = np.zeros(N_CLASSES, dtype=np.float32)
    for lbl in labels_str:
        lbl = lbl.strip()
        if lbl in label_to_idx:
            y[label_to_idx[lbl]] = 1.0
    label_map[row_id] = y

is_labeled = sc_meta["filename"].isin(labeled_files).values

def reshape_to_files(arr, meta):
    fnames = meta["filename"].values
    unique = list(dict.fromkeys(fnames))
    n_files = len(unique)
    D = arr.shape[1]
    out = np.zeros((n_files, N_WINDOWS, D), dtype=arr.dtype)
    file_to_idx = {f: i for i, f in enumerate(unique)}
    counters = np.zeros(n_files, dtype=int)
    for ri, fn in enumerate(fnames):
        fi = file_to_idx[fn]
        wi = counters[fi]
        if wi < N_WINDOWS:
            out[fi, wi] = arr[ri]
            counters[fi] += 1
    return out, unique

def parse_meta(fname):
    m = META_PAT.search(fname)
    if m is None:
        return 0, 0
    return int(m.group(1)), int(m.group(3))

# ── Labeled (66 files) ──
lab_meta = sc_meta[is_labeled].reset_index(drop=True)
lab_emb_flat = sc_emb[is_labeled]
lab_scores_flat = sc_scores[is_labeled]
lab_emb_files, lab_file_list = reshape_to_files(lab_emb_flat, lab_meta)
lab_scores_files, _ = reshape_to_files(lab_scores_flat, lab_meta)
lab_site_ids = np.array([parse_meta(fn)[0] for fn in lab_file_list], dtype=np.int64)
lab_hours = np.array([parse_meta(fn)[1] for fn in lab_file_list], dtype=np.int64)
lab_labels_files = np.zeros((len(lab_file_list), N_WINDOWS, N_CLASSES), dtype=np.float32)
for fi, fn in enumerate(lab_file_list):
    stem = Path(fn).stem
    for wi in range(N_WINDOWS):
        end_sec = (wi + 1) * WINDOW_SEC
        rid = f"{stem}_{end_sec}"
        if rid in label_map:
            lab_labels_files[fi, wi] = label_map[rid]
print(f"Labeled: {lab_emb_files.shape[0]} files")

# ── Unlabeled (~10,592 files) ──
unlab_mask = ~is_labeled
unlab_meta = sc_meta[unlab_mask].reset_index(drop=True)
unlab_emb_flat = sc_emb[unlab_mask]
unlab_scores_flat = sc_scores[unlab_mask]
unlab_emb_files, unlab_file_list = reshape_to_files(unlab_emb_flat, unlab_meta)
unlab_scores_files, _ = reshape_to_files(unlab_scores_flat, unlab_meta)
unlab_site_ids = np.array([parse_meta(fn)[0] for fn in unlab_file_list], dtype=np.int64)
unlab_hours = np.array([parse_meta(fn)[1] for fn in unlab_file_list], dtype=np.int64)
print(f"Unlabeled: {unlab_emb_files.shape[0]} files")

# ── Prior tables (computed from labeled set only — unchanged across rounds) ──
file_labels = (lab_labels_files.sum(axis=1) > 0).astype(np.float32)
global_p = file_labels.mean(axis=0).astype(np.float32)

prior_site_ids = sorted(set(int(s) for s in lab_site_ids))
site_to_pi = {s: i for i, s in enumerate(prior_site_ids)}
site_n = np.zeros(len(prior_site_ids), dtype=np.float32)
site_p = np.zeros((len(prior_site_ids), N_CLASSES), dtype=np.float32)
for s in prior_site_ids:
    m = (lab_site_ids == s)
    site_n[site_to_pi[s]] = m.sum()
    site_p[site_to_pi[s]] = file_labels[m].mean(axis=0)

prior_hours = sorted(set(int(h) for h in lab_hours))
hour_to_pi = {h: i for i, h in enumerate(prior_hours)}
hour_n = np.zeros(len(prior_hours), dtype=np.float32)
hour_p = np.zeros((len(prior_hours), N_CLASSES), dtype=np.float32)
for h in prior_hours:
    m = (lab_hours == h)
    hour_n[hour_to_pi[h]] = m.sum()
    hour_p[hour_to_pi[h]] = file_labels[m].mean(axis=0)

sh_to_pi = {}
sh_n_list, sh_p_list = [], []
for s in prior_site_ids:
    for h in prior_hours:
        m = (lab_site_ids == s) & (lab_hours == h)
        if m.sum() > 0:
            sh_to_pi[(s, h)] = len(sh_n_list)
            sh_n_list.append(float(m.sum()))
            sh_p_list.append(file_labels[m].mean(axis=0))
sh_n = np.array(sh_n_list, dtype=np.float32) if sh_n_list else np.zeros(0, dtype=np.float32)
sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, N_CLASSES), dtype=np.float32)

def compute_prior_logit(site_id, hour, eps=1e-4):
    p = global_p.astype(np.float32).copy()
    h_i = hour_to_pi.get(int(hour), -1)
    if h_i >= 0:
        nh = hour_n[h_i]
        wh = nh / (nh + PRIOR_STRENGTH_HOUR)
        p = wh * hour_p[h_i] + (1 - wh) * p
    s_i = site_to_pi.get(int(site_id), -1)
    if s_i >= 0:
        ns = site_n[s_i]
        ws = ns / (ns + PRIOR_STRENGTH_SITE)
        p = ws * site_p[s_i] + (1 - ws) * p
    sh_i = sh_to_pi.get((int(site_id), int(hour)), -1)
    if sh_i >= 0:
        nsh = sh_n[sh_i]
        wsh = nsh / (nsh + PRIOR_STRENGTH_SH)
        p = wsh * sh_p[sh_i] + (1 - wsh) * p
    p = np.clip(p, eps, 1 - eps)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)

lab_prior_files = np.stack(
    [compute_prior_logit(s, h) for s, h in zip(lab_site_ids, lab_hours)]
).astype(np.float32)
unlab_prior_files = np.stack(
    [compute_prior_logit(s, h) for s, h in zip(unlab_site_ids, unlab_hours)]
).astype(np.float32)

# Flat embedding for prototype init (labeled only)
lab_emb_flat_t = torch.tensor(lab_emb_flat, dtype=torch.float32).to(DEVICE)
lab_flat_labels_t = torch.zeros(len(lab_meta), N_CLASSES, dtype=torch.float32).to(DEVICE)
for i, rid in enumerate(lab_meta["row_id"]):
    if rid in label_map:
        lab_flat_labels_t[i] = torch.tensor(label_map[rid]).to(DEVICE)

print(f"Prior built: sites={len(prior_site_ids)}, hours={len(prior_hours)}, sh={len(sh_to_pi)}")

In [ ]:
# === PROTOSSM (NB3 v29) ===
class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.proj_delta = nn.Linear(d_model, d_model)
        self.proj_B = nn.Linear(d_model, d_state)
        self.proj_C = nn.Linear(d_model, d_state)
        self.proj_D = nn.Linear(d_model, d_model)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.log_A = nn.Parameter(torch.log(A))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B_sz, L, D = x.shape
        delta = F.softplus(self.proj_delta(x))
        B = self.proj_B(x)
        C = self.proj_C(x)
        D_param = self.proj_D(x)
        A = -torch.exp(self.log_A)
        h = torch.zeros(B_sz, self.d_model, self.d_state, device=x.device)
        outputs = []
        for t in range(L):
            dt = delta[:, t].unsqueeze(-1)
            dA = torch.exp(A.unsqueeze(0) * dt)
            dB = dt * B[:, t].unsqueeze(1)
            h = h * dA + x[:, t].unsqueeze(-1) * dB
            y = (h * C[:, t].unsqueeze(1)).sum(-1) + D_param[:, t]
            outputs.append(y)
        return self.dropout(torch.stack(outputs, dim=1))


class BiSSMBlock(nn.Module):
    def __init__(self, d_model, d_state, dropout=0.1):
        super().__init__()
        self.fwd_ssm = SelectiveSSM(d_model, d_state, dropout)
        self.bwd_ssm = SelectiveSSM(d_model, d_state, dropout)
        self.proj = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        fwd = self.fwd_ssm(x)
        bwd = self.bwd_ssm(x.flip(1)).flip(1)
        out = self.proj(torch.cat([fwd, bwd], dim=-1))
        return self.norm(x + out)


class CrossAttnBlock(nn.Module):
    def __init__(self, d_model, num_heads=2, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a, _ = self.attn(x, x, x, need_weights=False)
        return self.norm(x + self.dropout(a))


class ProtoSSM(nn.Module):
    def __init__(self, d_input, d_model, d_state, n_ssm_layers, n_classes,
                 n_windows, dropout=0.1, n_sites=32, meta_dim=8, n_attn_heads=2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_layers = nn.ModuleList([
            BiSSMBlock(d_model, d_state, dropout) for _ in range(n_ssm_layers)
        ])
        self.attn_layers = nn.ModuleList([
            CrossAttnBlock(d_model, num_heads=n_attn_heads, dropout=dropout)
            for _ in range(n_ssm_layers)
        ])
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.temperature = nn.Parameter(torch.tensor(10.0))
        self.bias = nn.Parameter(torch.zeros(n_classes))
        self.alpha = nn.Parameter(torch.ones(n_classes) * 0.5)

    def forward(self, emb, logits, site_ids=None, hours=None, prior_logit=None, lambda_prior=0.0):
        x = self.input_proj(emb) + self.pos_emb[:, :emb.shape[1]]
        if site_ids is not None and hours is not None:
            s_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            h_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([s_e, h_e], dim=-1))
            x = x + meta.unsqueeze(1)
        for ssm_layer, attn_layer in zip(self.ssm_layers, self.attn_layers):
            x = ssm_layer(x)
            x = attn_layer(x)
        x_norm = F.normalize(x, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        sim = torch.einsum("btd,cd->btc", x_norm, p_norm) * self.temperature + self.bias
        alpha = torch.sigmoid(self.alpha)
        out = alpha * sim + (1 - alpha) * logits
        if prior_logit is not None and lambda_prior > 0:
            out = out + lambda_prior * prior_logit.unsqueeze(1)
        return torch.sigmoid(out)

    def init_prototypes(self, emb_flat, labels_flat):
        with torch.no_grad():
            x = self.input_proj(emb_flat)
            for ci in range(self.prototypes.shape[0]):
                mask = labels_flat[:, ci] > 0.5
                if mask.sum() > 0:
                    self.prototypes[ci] = x[mask].mean(0)


# === MLP Head (NB2 v10) ===
class MLPHead(nn.Module):
    def __init__(self, d_input, d_hidden, n_classes, dropout=0.1,
                 n_sites=32, meta_dim=8):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_hidden)
        self.mlp = nn.Sequential(
            nn.Linear(d_hidden, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, n_classes),
        )
        self.temperature = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.ones(n_classes) * 0.5)

    def forward(self, emb, logits, site_ids=None, hours=None,
                prior_logit=None, lambda_prior=0.0):
        x = self.input_proj(emb)
        if site_ids is not None and hours is not None:
            s_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            h_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([s_e, h_e], dim=-1))
            x = x + meta.unsqueeze(1)
        h = self.mlp(x) * self.temperature
        alpha = torch.sigmoid(self.alpha)
        out = alpha * h + (1 - alpha) * logits
        if prior_logit is not None and lambda_prior > 0:
            out = out + lambda_prior * prior_logit.unsqueeze(1)
        return torch.sigmoid(out)


print("ProtoSSM + MLPHead defined.")

In [ ]:
# ─── Training & pseudo-label helpers ───
PSEUDO_WEIGHT = 0.3
N_ROUNDS = 5
BATCH_FILES = 64        # files per minibatch (avoid GPU OOM on 10k unlabeled)
PRED_BATCH = 128
WEIGHTS_DIR = Path("/kaggle/working/ns_weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)


# All training tensors live on CPU; per-batch slices move to GPU.
def train_one_seed(model_factory, seed, train_emb_np, train_logits_np, train_labels_np,
                   train_site_np, train_hour_np, train_prior_np, sample_weights_np,
                   teacher_prob_np, pos_weight_t):
    seed_everything(seed)
    model = model_factory().to(DEVICE)
    if hasattr(model, "init_prototypes"):
        model.init_prototypes(lab_emb_flat_t, lab_flat_labels_t)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    swa_model = AveragedModel(model) if USE_SWA else None
    swa_start = int(N_EPOCHS * SWA_START_FRAC)
    swa_n = 0
    best_loss = float("inf")
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    pw = pos_weight_t.unsqueeze(0).unsqueeze(0).to(DEVICE)   # (1, 1, C)
    N = train_emb_np.shape[0]

    for epoch in range(N_EPOCHS):
        model.train()
        perm = np.random.permutation(N)
        ep_losses = []
        for i in range(0, N, BATCH_FILES):
            idx = perm[i:i + BATCH_FILES]
            e   = torch.from_numpy(train_emb_np[idx]).to(DEVICE, non_blocking=True)
            lg  = torch.from_numpy(train_logits_np[idx]).to(DEVICE, non_blocking=True)
            lbl = torch.from_numpy(train_labels_np[idx]).to(DEVICE, non_blocking=True)
            st  = torch.from_numpy(train_site_np[idx]).long().to(DEVICE, non_blocking=True)
            hr  = torch.from_numpy(train_hour_np[idx]).long().to(DEVICE, non_blocking=True)
            pr  = torch.from_numpy(train_prior_np[idx]).to(DEVICE, non_blocking=True)
            sw  = torch.from_numpy(sample_weights_np[idx]).to(DEVICE, non_blocking=True)
            sw  = sw.unsqueeze(1).unsqueeze(2)               # (B, 1, 1)
            tp  = torch.from_numpy(teacher_prob_np[idx]).to(DEVICE, non_blocking=True)

            out = model(e, lg, site_ids=st, hours=hr,
                        prior_logit=pr, lambda_prior=LAMBDA_PRIOR)
            bce = F.binary_cross_entropy(out, lbl, reduction="none")
            loss_main = (bce * pw * sw).mean()
            loss_kd = F.binary_cross_entropy(out, tp, reduction="mean")
            loss = loss_main + LAMBDA_KD * loss_kd

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_losses.append(loss.item())

            del e, lg, lbl, st, hr, pr, sw, tp, out, loss

        scheduler.step()
        ep_loss = float(np.mean(ep_losses)) if ep_losses else float('inf')
        if ep_loss < best_loss:
            best_loss = ep_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if USE_SWA and epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_n += 1

    if USE_SWA and swa_n >= 1:
        model.load_state_dict(swa_model.module.state_dict())
    else:
        model.load_state_dict(best_state)
    model.eval()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return model, best_loss


@torch.no_grad()
def predict_files_cpu(models, emb_np, logits_np, site_np, hour_np, prior_np,
                      batch=PRED_BATCH):
    # Returns CPU numpy array of shape (N, N_WINDOWS, N_CLASSES) — average over models.
    N = emb_np.shape[0]
    out_sum = np.zeros((N, N_WINDOWS, N_CLASSES), dtype=np.float32)
    for m in models:
        m.eval()
        for i in range(0, N, batch):
            j = min(i + batch, N)
            e  = torch.from_numpy(emb_np[i:j]).to(DEVICE, non_blocking=True)
            lg = torch.from_numpy(logits_np[i:j]).to(DEVICE, non_blocking=True)
            st = torch.from_numpy(site_np[i:j]).long().to(DEVICE, non_blocking=True)
            hr = torch.from_numpy(hour_np[i:j]).long().to(DEVICE, non_blocking=True)
            pr = torch.from_numpy(prior_np[i:j]).to(DEVICE, non_blocking=True)
            o = m(e, lg, site_ids=st, hours=hr,
                  prior_logit=pr, lambda_prior=LAMBDA_PRIOR)
            out_sum[i:j] += o.detach().cpu().numpy()
            del e, lg, st, hr, pr, o
    out_sum /= len(models)
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return out_sum


print(f"NS helpers: BATCH_FILES={BATCH_FILES}, PSEUDO_WEIGHT={PSEUDO_WEIGHT}, "
      f"N_ROUNDS={N_ROUNDS}, SEEDS={SEEDS}")

In [ ]:
# ─── 5-round Noisy Student loop ───
# All training data lives on CPU as numpy arrays (large unlabeled set ~782MB).
# Per-batch slices are moved to GPU inside train_one_seed.

# Numpy views on labeled set (already in float32 from LOAD cell)
lab_emb_np    = lab_emb_files
lab_logits_np = lab_scores_files
lab_labels_np = lab_labels_files
lab_site_np   = lab_site_ids
lab_hour_np   = lab_hours
lab_prior_np  = lab_prior_files

# Numpy views on unlabeled set
unlab_emb_np    = unlab_emb_files
unlab_logits_np = unlab_scores_files
unlab_site_np   = unlab_site_ids
unlab_hour_np   = unlab_hours
unlab_prior_np  = unlab_prior_files

# pos_weight from labeled distribution
_lab_lbl_t = torch.from_numpy(lab_labels_np)
_pc = _lab_lbl_t.sum(dim=(0, 1)).clamp(min=1)
_nc = _lab_lbl_t.shape[0] * _lab_lbl_t.shape[1] - _pc
pos_weight_t = (_nc / _pc).clamp(max=30.0)
del _lab_lbl_t

# Perch teacher (sigmoid of mapped logits) on labeled set
teacher_prob_lab_np = (1.0 / (1.0 + np.exp(-lab_logits_np))).clip(1e-7, 1 - 1e-7).astype(np.float32)

current_pseudo_np = None  # (N_unlab, N_WINDOWS, N_CLASSES) after round 0

for r in range(N_ROUNDS):
    print(f"\\n========== ROUND {r} ==========")
    t_round = time.time()

    if r == 0 or current_pseudo_np is None:
        train_emb_np    = lab_emb_np
        train_logits_np = lab_logits_np
        train_labels_np = lab_labels_np
        train_site_np   = lab_site_np
        train_hour_np   = lab_hour_np
        train_prior_np  = lab_prior_np
        sample_w_np     = np.ones(lab_emb_np.shape[0], dtype=np.float32)
        teacher_prob_np = teacher_prob_lab_np
    else:
        train_emb_np    = np.concatenate([lab_emb_np,    unlab_emb_np],    axis=0)
        train_logits_np = np.concatenate([lab_logits_np, unlab_logits_np], axis=0)
        train_labels_np = np.concatenate([lab_labels_np, current_pseudo_np.astype(np.float32)], axis=0)
        train_site_np   = np.concatenate([lab_site_np,   unlab_site_np],   axis=0)
        train_hour_np   = np.concatenate([lab_hour_np,   unlab_hour_np],   axis=0)
        train_prior_np  = np.concatenate([lab_prior_np,  unlab_prior_np],  axis=0)
        sample_w_np = np.concatenate([
            np.ones(lab_emb_np.shape[0], dtype=np.float32),
            np.full(unlab_emb_np.shape[0], PSEUDO_WEIGHT, dtype=np.float32),
        ], axis=0)
        teacher_prob_np = np.concatenate([
            teacher_prob_lab_np,
            current_pseudo_np.astype(np.float32),
        ], axis=0)

    print(f"  train data: {train_emb_np.shape[0]} files "
          f"({train_emb_np.nbytes/1e9:.2f} GB on CPU)")

    # ── Train ProtoSSM × SEEDS ──
    proto_models = []
    for seed in SEEDS:
        factory = lambda: ProtoSSM(
            d_input=1536, d_model=D_MODEL, d_state=D_STATE,
            n_ssm_layers=N_SSM_LAYERS, n_classes=N_CLASSES,
            n_windows=N_WINDOWS, dropout=DROPOUT,
            n_sites=N_SITES, meta_dim=META_DIM,
        )
        t_seed = time.time()
        m, bl = train_one_seed(factory, seed,
                               train_emb_np, train_logits_np, train_labels_np,
                               train_site_np, train_hour_np, train_prior_np,
                               sample_w_np, teacher_prob_np, pos_weight_t)
        proto_models.append(m)
        torch.save(m.state_dict(), WEIGHTS_DIR / f"round{r}_proto_seed{seed}.pt")
        print(f"  ProtoSSM[seed {seed}] best_loss={bl:.4f}, {time.time()-t_seed:.0f}s")

    # ── Train MLPHead × SEEDS ──
    mlp_models = []
    for seed in SEEDS:
        factory = lambda: MLPHead(
            d_input=1536, d_hidden=MLP_HIDDEN, n_classes=N_CLASSES,
            dropout=DROPOUT, n_sites=N_SITES, meta_dim=META_DIM,
        )
        t_seed = time.time()
        m, bl = train_one_seed(factory, seed,
                               train_emb_np, train_logits_np, train_labels_np,
                               train_site_np, train_hour_np, train_prior_np,
                               sample_w_np, teacher_prob_np, pos_weight_t)
        mlp_models.append(m)
        torch.save(m.state_dict(), WEIGHTS_DIR / f"round{r}_mlp_seed{seed}.pt")
        print(f"  MLPHead[seed {seed}] best_loss={bl:.4f}, {time.time()-t_seed:.0f}s")

    # ── Pseudo-label for next round ──
    if r < N_ROUNDS - 1:
        proto_pred = predict_files_cpu(proto_models,
                                       unlab_emb_np, unlab_logits_np,
                                       unlab_site_np, unlab_hour_np, unlab_prior_np)
        mlp_pred = predict_files_cpu(mlp_models,
                                     unlab_emb_np, unlab_logits_np,
                                     unlab_site_np, unlab_hour_np, unlab_prior_np)
        current_pseudo_np = ((proto_pred + mlp_pred) / 2.0).clip(1e-3, 1 - 1e-3).astype(np.float32)
        print(f"  pseudo updated: shape={current_pseudo_np.shape}, "
              f"mean={current_pseudo_np.mean():.4f}, max={current_pseudo_np.max():.4f}")
        del proto_pred, mlp_pred

    # Free models for next round (weights already on disk)
    del proto_models, mlp_models
    if r > 0:
        del train_emb_np, train_logits_np, train_labels_np
        del train_site_np, train_hour_np, train_prior_np
        del sample_w_np, teacher_prob_np
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    print(f"  Round {r} done in {(time.time()-t_round)/60:.1f} min")

print(f"\\n=== ALL ROUNDS DONE ===")
print(f"Saved weights to: {WEIGHTS_DIR}")

In [ ]:
# ─── Verify ───
saved = sorted(WEIGHTS_DIR.glob("*.pt"))
print(f"Total weight files: {len(saved)} (expected {N_ROUNDS * len(SEEDS) * 2} = 5 rounds × 5 seeds × 2 models)")
total_mb = sum(p.stat().st_size for p in saved) / 1e6
print(f"Total size: {total_mb:.1f} MB")
for p in saved[:5]:
    print(f"  {p.name}: {p.stat().st_size/1e6:.2f} MB")
print("  ...")
for p in saved[-5:]:
    print(f"  {p.name}: {p.stat().st_size/1e6:.2f} MB")